In [55]:
import pandas as pd

In [56]:
df = pd.read_csv('diabetes (1).csv')

In [57]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [58]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [59]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

In [60]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [61]:
X = scaler.fit_transform(X)

In [62]:
X.shape

(768, 8)

In [63]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [64]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Dropout

In [65]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=8))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [66]:
model.fit(X_train, y_train, batch_size=32, epochs=100, validation_data=(X_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6401 - loss: 0.6684 - val_accuracy: 0.7338 - val_loss: 0.6026
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6987 - loss: 0.6227 - val_accuracy: 0.7532 - val_loss: 0.5671
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7166 - loss: 0.5895 - val_accuracy: 0.7597 - val_loss: 0.5409
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7313 - loss: 0.5646 - val_accuracy: 0.7792 - val_loss: 0.5213
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7394 - loss: 0.5446 - val_accuracy: 0.8182 - val_loss: 0.5056
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7410 - loss: 0.5288 - val_accuracy: 0.8182 - val_loss: 0.4925
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7492 - loss: 0.5158 - val_accuracy: 0.8117 - val_loss: 0.4831
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7492 - loss: 0.5056 - val_accuracy: 0.7987 

In [68]:
# 1. How to select appropriate optimizer
# 2. No. of nodes in a layer
# 3. How to select no. of layers
# 4. All in one model

# 1. How to Select appropriate Optimizer

In [69]:
!pip install keras-tuner
import keras_tuner as kt

In [70]:
def build_model(hp):
    model = Sequential()
    model.add(Dense(32, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop', 'adadelta'])

    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [71]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [72]:
tuner.search(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

In [73]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [74]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [75]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [76]:
model.fit(X_train, y_train, batch_size=32, epochs=100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7508 - loss: 0.5347 - val_accuracy: 0.7922 - val_loss: 0.4959
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7622 - loss: 0.5155 - val_accuracy: 0.7792 - val_loss: 0.4799
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7687 - loss: 0.5019 - val_accuracy: 0.7597 - val_loss: 0.4708
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7638 - loss: 0.4929 - val_accuracy: 0.7792 - val_loss: 0.4628
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7638 - loss: 0.4850 - val_accuracy: 0.7792 - val_loss: 0.4587
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7671 - loss: 0.4791 - val_accuracy: 0.7792 - val_loss: 0.4548
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4741 - val_accuracy: 0.7792 - val_loss: 0.4536
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4702 - val_accuracy: 0.77

# 2. No. of Neurons in Layer

In [77]:
def build_model(hp):
    model = Sequential()
    units = hp.Int('units', min_value=8, max_value=128, step=8)

    model.add(Dense(units=units, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [78]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5,
                        directory='mydir9')

Reloading Tuner from mydir9/untitled_project/tuner0.json


In [79]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [80]:
tuner.get_best_hyperparameters()[0].values

{'units': 80}

In [81]:
model = tuner.get_best_models(num_models=1)[0]

In [82]:
model.fit(X_train, y_train, batch_size=32, epochs=100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7736 - loss: 0.4877 - val_accuracy: 0.7987 - val_loss: 0.4772
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7752 - loss: 0.4742 - val_accuracy: 0.7987 - val_loss: 0.4755
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7752 - loss: 0.4665 - val_accuracy: 0.8052 - val_loss: 0.4686
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7769 - loss: 0.4599 - val_accuracy: 0.8052 - val_loss: 0.4667
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7769 - loss: 0.4563 - val_accuracy: 0.8052 - val_loss: 0.4664
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7801 - loss: 0.4526 - val_accuracy: 0.8052 - val_loss: 0.4646
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7736 - loss: 0.4508 - val_accuracy: 0.8117 - val_loss: 0.4632
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7801 - loss: 0.4479 - val_accuracy: 0.79

# 3. How to select No of layers

In [83]:
def build_model(hp):
    model = Sequential()
    model.add(Dense(88, activation='relu', input_dim=8))

    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):

        model.add(Dense(88, activation='relu', input_dim=8))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [84]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5,
                        directory='mydir10')

Reloading Tuner from mydir10/untitled_project/tuner0.json


In [85]:
tuner.search(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

In [86]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 9}

In [87]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 46 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [88]:
model.fit(X_train, y_train, batch_size=32, epochs=100, initial_epoch=11, validation_data=(X_test, y_test))

Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.7671 - loss: 0.4866 - val_accuracy: 0.8052 - val_loss: 0.4815
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7818 - loss: 0.4597 - val_accuracy: 0.8182 - val_loss: 0.4529
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7899 - loss: 0.4489 - val_accuracy: 0.8182 - val_loss: 0.4468
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7980 - loss: 0.4286 - val_accuracy: 0.8247 - val_loss: 0.4770
Epoch 16/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7997 - loss: 0.4226 - val_accuracy: 0.8117 - val_loss: 0.4551
Epoch 17/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8143 - loss: 0.4008 - val_accuracy: 0.8182 - val_loss: 0.4471
Epoch 18/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7980 - loss: 0.4427 - val_accuracy: 0.8052 - val_loss: 0.4663
Epoch 19/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7997 - loss: 0.4072 - val_accuracy: 0

# 4. All in one model

In [89]:
def build_model(hp):
  model = Sequential()

  counter = 0
  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):

    if counter == 0:
      model.add(
          Dense(
              hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
              activation= hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid']),
              input_dim=8
          )
      )
      model.add(Dropout(hp.Choice('dropout' + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])))

    else:
      model.add(
          Dense(
              hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
              activation= hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid'])
          )
      )
      model.add(Dropout(hp.Choice('dropout' + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])))
    counter += 1

  model.add(Dense(1, activation='sigmoid'))

  model.compile(optimizer=hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop', 'adadelta']), loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [90]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5, directory='mydir11')

Reloading Tuner from mydir11/untitled_project/tuner0.json


In [91]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [92]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 4,
 'units0': 104,
 'activation0': 'relu',
 'optimizer': 'adam',
 'units1': 112,
 'activation1': 'sigmoid',
 'units2': 40,
 'activation2': 'tanh',
 'units3': 48,
 'activation3': 'tanh',
 'units4': 72,
 'activation4': 'tanh',
 'units5': 112,
 'activation5': 'tanh',
 'units6': 104,
 'activation6': 'relu',
 'units7': 72,
 'activation7': 'relu'}

In [93]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print(best_hp.values)

{'num_layers': 4, 'units0': 104, 'activation0': 'relu', 'optimizer': 'adam', 'units1': 112, 'activation1': 'sigmoid', 'units2': 40, 'activation2': 'tanh', 'units3': 48, 'activation3': 'tanh', 'units4': 72, 'activation4': 'tanh', 'units5': 112, 'activation5': 'tanh', 'units6': 104, 'activation6': 'relu', 'units7': 72, 'activation7': 'relu'}


In [94]:
tuner.results_summary()

Results summary
Results in mydir11/untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 4 summary
Hyperparameters:
num_layers: 4
units0: 104
activation0: relu
optimizer: adam
units1: 112
activation1: sigmoid
units2: 40
activation2: tanh
units3: 48
activation3: tanh
units4: 72
activation4: tanh
units5: 112
activation5: tanh
units6: 104
activation6: relu
units7: 72
activation7: relu
Score: 0.8116883039474487

Trial 0 summary
Hyperparameters:
num_layers: 8
units0: 120
activation0: relu
optimizer: adam
units1: 8
activation1: relu
units2: 8
activation2: relu
units3: 8
activation3: relu
units4: 8
activation4: relu
units5: 8
activation5: relu
units6: 8
activation6: relu
units7: 8
activation7: relu
Score: 0.7857142686843872

Trial 3 summary
Hyperparameters:
num_layers: 6
units0: 88
activation0: sigmoid
optimizer: adam
units1: 72
activation1: relu
units2: 104
activation2: relu
units3: 72
activation3: tanh
units4: 64
activation4: sigmoid
units5: 8
activa

In [95]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [96]:
model.fit(X_train, y_train, batch_size=32, epochs=200, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.7541 - loss: 0.4918 - val_accuracy: 0.7792 - val_loss: 0.4634
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7769 - loss: 0.4881 - val_accuracy: 0.8052 - val_loss: 0.4761
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7704 - loss: 0.4908 - val_accuracy: 0.8052 - val_loss: 0.4675
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7899 - loss: 0.4719 - val_accuracy: 0.7727 - val_loss: 0.4595
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7785 - loss: 0.4717 - val_accuracy: 0.7857 - val_loss: 0.4694
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7704 - loss: 0.4818 - val_accuracy: 0.8117 - val_loss: 0.4543
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7720 - loss: 0.4722 - val_accuracy: 0.8117 - val_loss: 0.4575
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7720 - loss: 0.4726 - val_accuracy: 